# Лабораторная работа 7: классификация текстов (20 newsgroups) с помощью SVM и TF-IDF

In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.feature_extraction import text

## 1. Загрузите объекты из новостного датасета 20 newsgroups, относящиеся к категориям "космос" и "атеизм" (инструкция приведена выше).

## 2. Вычислите TF-IDF-признаки для всех текстов. Обратите внимание, что в этом задании мы предлагаем вам вычислить TF-IDF по всем данным. При таком подходе получается, что признаки на обучающем множестве используют информацию из тестовой выборки — но такая ситуация вполне законна, поскольку мы не используем значения целевой переменной из теста. На практике нередко встречаются ситуации, когда признаки объектов тестовой выборки известны на момент обучения, и поэтому можно ими пользоваться при обучении алгоритма.

In [2]:
newsgroups = datasets.fetch_20newsgroups(subset='all', categories=['alt.atheism', 'sci.space'])
vectorizer = text.TfidfVectorizer()
X = vectorizer.fit_transform(newsgroups.data)
y = newsgroups.target

In [3]:
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.svm import SVC

## 3. Подберите минимальный лучший параметр C из множества \([10^{-5}, 10^{-4}, \dots 10^4, 10^5]\) для SVM с линейным ядром (kernel='linear') при помощи кросс-валидации по 5 блокам. Укажите параметр random_state=241 и для SVM, и для KFold. В качестве меры качества используйте долю верных ответов (accuracy).

In [4]:
grid = {'C': np.power(10.0, np.arange(-5, 6))}
cv = KFold(n_splits=5, shuffle=True, random_state= 241)
clf = SVC(kernel='linear', random_state=241)
gs = GridSearchCV(clf, grid, scoring='accuracy', cv=cv)
gs.fit(X, y)

GridSearchCV(cv=KFold(n_splits=5, random_state=241, shuffle=True),
             estimator=SVC(kernel='linear', random_state=241),
             param_grid={'C': array([1.e-05, 1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02,
       1.e+03, 1.e+04, 1.e+05])},
             scoring='accuracy')

In [5]:
C_best = gs.best_params_['C']

## 4. Обучите SVM по всей выборке с лучшим параметром C, найденным на предыдущем шаге.

In [6]:
best_svc = SVC(C=C_best, random_state=241, kernel='linear').fit(X, y)

## 5. Найдите 10 слов с наибольшим по модулю весом. Они являются ответом на это задание. Укажите их через запятую, в нижнем регистре, в лексикографическом порядке.

In [7]:
coef = best_svc.coef_.toarray()[0]
feature_names = vectorizer.get_feature_names_out()

word_weights = [(word, abs(weight)) for word, weight in zip(feature_names, coef)]
word_weights.sort(key=lambda x: x[1], reverse=True)

top_10_words = [word for word, weight in word_weights[:10]]
top_10_words_sorted = sorted(top_10_words)

ans = ', '.join(top_10_words_sorted)

In [8]:
with open('1.txt', 'w') as file:
    file.write(f'{ans}')